In [1]:
import polars as pl
import numpy as np
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import os
from collections import defaultdict
import datetime as dt
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mticker
sys.path.insert(1, os.path.abspath(".."))
os.chdir(os.path.abspath('..')) # Change the workdir to the parent folder

# ── Visual defaults ───────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
SIRS_COLORS = {0: "#4CAF50", 1: "#FFC107", 2: "#FF9800", 3: "#F44336", 4: "#9C27B0"}

In [2]:
from src.configs.dataconfig import input_filenames_v3, input_output_config_3, input_filenames_v2, input_output_config_2
from src.dataloader import DataLoader

In [ ]:
df_all_v2 = DataLoader(input_output_config_2).load_data()
df_all_v3 = DataLoader(input_output_config_3).load_data()

In [ ]:
PROJECT_V3 = 'phase3_v3'
PROJECT_V2 = 'raw_data_phase3_v2_old'

### Load output data

In [ ]:
# v3 
df_sepsis_3_v3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_sepsis3.parquet'
	)
)

df_sepsis_2_v3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_sepsis2.parquet'
	)
)

df_sepsis_1_v3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_sepsis1.parquet'
	)
)

df_sirs_3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_sirs_postagg.parquet'
	)
)

df_organdysf_3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_organdysfunction.parquet'
	)
)

df_o2vent_3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_o2vent.parquet'
	)
)

df_shock_3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path), PROJECT_V3, 'df_septicshock.parquet'
	)
)

df_infect_long_3 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V3,'df_infect_long.parquet'
	)
)

In [ ]:
# v2 
df_sepsis_3_v2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V2,'df_sepsis3.parquet'
    )
)
df_sepsis_2_v2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V2,'df_sepsis2.parquet'
    )
)
df_sepsis_1_v2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V2,'df_sepsis1.parquet'
    )
)


df_sirs_2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path), PROJECT_V2,'df_sirs.parquet'
	)
)

df_organdysf_2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path), PROJECT_V2,'df_organdysfunction.parquet'
	)
)

df_o2vent_2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V2,'df_agg_vent.parquet'
	)
)

df_shock_2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path), PROJECT_V2, 'df_septicshock.parquet'
	)
)

df_infect_long_2 = pl.read_parquet(
    os.path.join(
        os.path.dirname(input_output_config_3.output_path),PROJECT_V2,'df_infect_long.parquet'
	)
)


### Concatenate 

In [ ]:
cols1 = df_sepsis_1_v3.columns
cols2 = df_sepsis_1_v3.columns
cols3 = df_sepsis_1_v3.columns
idx = cols2.index('earliest_sepsis1_instance')
cols2[idx] = 'earliest_sepsis2_instance'
cols3[idx] = 'earliest_sepsis3_instance'
cols2.remove('sirs_score')
cols3.remove('sirs_score')
cols1.remove('sirs_score')

In [ ]:
df_sepsis_1_v3 = df_sepsis_1_v3.select(
    cols1
).with_columns(
    pl.lit(1).alias('sepsis_score')
).rename({"earliest_sepsis1_instance": "sepsis_instance"})

df_sepsis_2_v3 = df_sepsis_2_v3.select(
    cols2
).with_columns(
    pl.lit(2).alias('sepsis_score')
).rename({"earliest_sepsis2_instance": "sepsis_instance"})

df_sepsis_3_v3 = df_sepsis_3_v3.select(
    cols3
).with_columns(
    pl.lit(3).alias('sepsis_score')
).rename({"earliest_sepsis3_instance": "sepsis_instance"})

df_sepsis_v3 = pl.concat([df_sepsis_1_v3, df_sepsis_2_v3, df_sepsis_3_v3], how='vertical')

In [ ]:
df_sepsis_1_v2 = df_sepsis_1_v2.select(
    cols1
).with_columns(
    pl.lit(1).alias('sepsis_score')
).rename({"earliest_sepsis1_instance": "sepsis_instance"})

df_sepsis_2_v2 = df_sepsis_2_v2.select(
    cols2
).with_columns(
    pl.lit(2).alias('sepsis_score')
).rename({"earliest_sepsis2_instance": "sepsis_instance"})

df_sepsis_3_v2 = df_sepsis_3_v2.select(
    cols3
).with_columns(
    pl.lit(3).alias('sepsis_score')
).rename({"earliest_sepsis3_instance": "sepsis_instance"})

df_sepsis_v2 = pl.concat([df_sepsis_1_v2, df_sepsis_2_v2, df_sepsis_3_v2], how='vertical')

### Conflict between v2 and v3 

In [ ]:
# First sepsis onset v3
df_sepsis_v3_firstonset = df_sepsis_v3.join(
    df_sepsis_v3.group_by("EncounterEpicCsn").agg(
    pl.col("sepsis_instance").min().alias('first_sepsis_instant')
    ),
	on='EncounterEpicCsn',
	how='inner'
).filter(
    pl.col("sepsis_instance")==pl.col("first_sepsis_instant")
).group_by(
    "EncounterEpicCsn", "first_sepsis_instant"
).agg(
    pl.col("sepsis_score").max(),
    pl.col("infect_dt").min().alias('min_infect_dt'),
    pl.col("infect_dt").max().alias('max_infect_dt'),
    pl.col("Event_DateTime").min().alias('min_Event_DateTime'),
    pl.col("Event_DateTime").max().alias('max_Event_DateTime'),
).sort(by=["EncounterEpicCsn", 'first_sepsis_instant'])
df_sepsis_v3_firstonset

EncounterEpicCsn,first_sepsis_instant,sepsis_score,min_infect_dt,max_infect_dt,min_Event_DateTime,max_Event_DateTime
i64,datetime[μs],i32,datetime[μs],datetime[μs],datetime[μs],datetime[μs]
659308243,2022-06-14 16:40:00,2,2022-06-14 22:10:00,2022-06-15 23:40:38,2022-06-14 16:40:00,2022-06-14 16:40:00
660956097,2022-06-27 23:58:00,2,2022-06-28 06:10:00,2022-06-28 10:39:00,2022-06-27 23:58:00,2022-06-27 23:58:00
661424247,2022-08-28 04:56:00,1,2022-08-28 05:30:00,2022-08-28 10:56:00,2022-08-28 04:56:00,2022-08-28 04:56:00
661471289,2022-06-07 12:00:00,2,2022-06-07 19:47:00,2022-06-08 21:58:00,2022-06-07 12:00:00,2022-06-07 12:00:00
662152913,2022-08-15 14:10:00,1,2022-08-15 16:45:00,2022-08-16 01:55:00,2022-08-15 14:10:00,2022-08-15 14:10:00
…,…,…,…,…,…,…
753558469,2026-05-24 18:39:00,2,2026-05-24 18:39:00,2026-05-24 18:39:00,2026-05-24 18:40:00,2026-05-25 16:00:00
753563196,2026-05-25 04:49:00,2,2026-05-25 04:49:00,2026-05-25 04:49:00,2026-05-25 05:17:00,2026-05-25 08:00:00
753573877,2026-05-25 20:45:00,1,2026-05-25 21:53:00,2026-05-26 13:06:06,2026-05-25 20:45:00,2026-05-25 20:45:00


In [ ]:
# Maximum sepsis onset v3
df_sepsis_v3_maxonset = df_sepsis_v3.join(
    df_sepsis_v3.group_by("EncounterEpicCsn").agg(
    pl.col("sepsis_score").max().alias('max_sepsis_score')
    ),
	on='EncounterEpicCsn',
	how='inner'
).filter(
    pl.col("sepsis_score")==pl.col("max_sepsis_score")
).group_by(
    "EncounterEpicCsn", "max_sepsis_score"
).agg(
    pl.col("sepsis_instance").min().alias('sepsis_instant'),
    pl.col("infect_dt").min().alias('min_infect_dt'),
    pl.col("Event_DateTime").min().alias('min_Event_DateTime'),
).sort(by=["EncounterEpicCsn", 'sepsis_instant'])
df_sepsis_v3_maxonset

EncounterEpicCsn,max_sepsis_score,sepsis_instant,min_infect_dt,min_Event_DateTime
i64,i32,datetime[μs],datetime[μs],datetime[μs]
659308243,3,2022-06-14 22:10:00,2022-06-14 22:10:00,2022-06-15 01:22:03
660956097,3,2022-06-28 03:59:07,2022-06-28 06:10:00,2022-06-28 03:59:07
661424247,3,2022-08-28 05:30:00,2022-08-28 05:30:00,2022-08-28 12:04:34
661471289,2,2022-06-07 12:00:00,2022-06-07 19:47:00,2022-06-07 12:00:00
662152913,2,2022-08-15 16:45:00,2022-08-15 16:45:00,2022-08-15 16:45:00
…,…,…,…,…
753558469,3,2026-05-25 19:43:52,2026-05-25 19:43:52,2026-05-27 12:24:00
753563196,2,2026-05-25 04:49:00,2026-05-25 04:49:00,2026-05-25 05:22:00
753573877,1,2026-05-25 20:45:00,2026-05-25 21:53:00,2026-05-25 20:45:00


In [ ]:
# First sepsis onset v12
df_sepsis_v2_firstonset = df_sepsis_v2.join(
    df_sepsis_v2.group_by("EncounterEpicCsn").agg(
    pl.col("sepsis_instance").min().alias('first_sepsis_instant')
    ),
	on='EncounterEpicCsn',
	how='inner'
).filter(
    pl.col("sepsis_instance")==pl.col("first_sepsis_instant")
).group_by(
    "EncounterEpicCsn", "first_sepsis_instant"
).agg(
    pl.col("sepsis_score").max(),
    pl.col("infect_dt").min().alias('min_infect_dt'),
    pl.col("infect_dt").max().alias('max_infect_dt'),
    pl.col("Event_DateTime").min().alias('min_Event_DateTime'),
    pl.col("Event_DateTime").max().alias('max_Event_DateTime'),
).sort(by=["EncounterEpicCsn", 'first_sepsis_instant'])
df_sepsis_v2_firstonset

EncounterEpicCsn,first_sepsis_instant,sepsis_score,min_infect_dt,max_infect_dt,min_Event_DateTime,max_Event_DateTime
i64,datetime[μs],i32,datetime[μs],datetime[μs],datetime[μs],datetime[μs]
659308243,2022-06-14 17:52:00,3,2022-06-14 22:10:00,2022-06-14 22:10:00,2022-06-14 17:52:00,2022-06-14 17:52:00
660956097,2022-06-27 18:28:00,1,2022-06-28 06:10:00,2022-06-28 06:10:00,2022-06-27 18:28:00,2022-06-27 18:28:00
661424247,2022-08-28 02:18:00,3,2022-08-28 05:30:00,2022-08-28 05:30:00,2022-08-28 02:18:00,2022-08-28 02:18:00
661471289,2022-06-07 12:07:00,2,2022-06-07 19:47:00,2022-06-07 19:47:00,2022-06-07 12:07:00,2022-06-07 12:07:00
662152913,2022-08-15 14:10:00,1,2022-08-15 16:45:00,2022-08-15 16:45:00,2022-08-15 14:10:00,2022-08-15 14:10:00
…,…,…,…,…,…,…
753558469,2026-05-24 18:39:00,2,2026-05-24 18:39:00,2026-05-24 18:39:00,2026-05-24 18:49:00,2026-05-24 18:54:00
753563196,2026-05-25 04:42:00,1,2026-05-25 04:49:00,2026-05-25 04:49:00,2026-05-25 04:42:00,2026-05-25 04:42:00
753573877,2026-05-25 16:27:00,1,2026-05-25 21:53:00,2026-05-25 21:53:00,2026-05-25 16:27:00,2026-05-25 16:27:00


In [ ]:
# Maximum sepsis onset v2
df_sepsis_v2_maxonset = df_sepsis_v2.join(
    df_sepsis_v2.group_by("EncounterEpicCsn").agg(
    pl.col("sepsis_score").max().alias('max_sepsis_score')
    ),
	on='EncounterEpicCsn',
	how='inner'
).filter(
    pl.col("sepsis_score")==pl.col("max_sepsis_score")
).group_by(
    "EncounterEpicCsn", "max_sepsis_score"
).agg(
    pl.col("sepsis_instance").min().alias('sepsis_instant'),
    pl.col("infect_dt").min().alias('min_infect_dt'),
    pl.col("Event_DateTime").min().alias('min_Event_DateTime'),
).sort(by=["EncounterEpicCsn", 'sepsis_instant'])
df_sepsis_v2_maxonset

EncounterEpicCsn,max_sepsis_score,sepsis_instant,min_infect_dt,min_Event_DateTime
i64,i32,datetime[μs],datetime[μs],datetime[μs]
659308243,3,2022-06-14 17:52:00,2022-06-14 22:10:00,2022-06-14 17:52:00
660956097,3,2022-06-28 01:09:00,2022-06-28 06:10:00,2022-06-28 01:09:00
661424247,3,2022-08-28 02:18:00,2022-08-28 05:30:00,2022-08-28 02:18:00
661471289,2,2022-06-07 12:07:00,2022-06-07 19:47:00,2022-06-07 12:07:00
662152913,2,2022-08-15 16:45:00,2022-08-15 16:45:00,2022-08-15 17:00:00
…,…,…,…,…
753558469,2,2026-05-24 18:39:00,2026-05-24 18:39:00,2026-05-24 18:54:00
753563196,2,2026-05-25 04:49:00,2026-05-25 04:49:00,2026-05-25 05:24:00
753573877,1,2026-05-25 16:27:00,2026-05-25 21:53:00,2026-05-25 16:27:00


In [ ]:
# Focusing only on the encounter categorized by both v2 and v3
df_sepsis_v23_maxonset = df_sepsis_v2_maxonset.join(
    df_sepsis_v3_maxonset, on=['EncounterEpicCsn'], how='inner', suffix='_v3'
)
# df_sepsis_v23_maxonset
df_sepsis_v23_maxonset.filter(
    pl.col("max_sepsis_score_v3")!=pl.col("max_sepsis_score")
).sort(by=['EncounterEpicCsn'])

EncounterEpicCsn,max_sepsis_score,sepsis_instant,min_infect_dt,min_Event_DateTime,max_sepsis_score_v3,sepsis_instant_v3,min_infect_dt_v3,min_Event_DateTime_v3
i64,i32,datetime[μs],datetime[μs],datetime[μs],i32,datetime[μs],datetime[μs],datetime[μs]
662608744,2,2022-07-13 11:50:00,2022-07-13 11:50:00,2022-07-13 13:03:00,3,2022-07-13 11:50:00,2022-07-13 11:50:00,2022-07-14 14:18:00
662965478,2,2022-06-01 15:28:00,2022-06-01 15:28:00,2022-06-01 18:42:00,3,2022-06-01 15:28:00,2022-06-01 15:28:00,2022-06-01 22:40:00
663195289,2,2022-06-06 17:37:00,2022-06-06 17:54:00,2022-06-06 17:37:00,3,2022-06-06 17:54:00,2022-06-06 17:54:00,2022-06-07 06:05:38
663236716,2,2022-06-07 12:30:00,2022-06-07 14:44:00,2022-06-07 12:30:00,3,2022-06-07 14:44:00,2022-06-07 14:44:00,2022-06-08 04:31:00
663380959,2,2022-07-21 06:30:00,2022-07-23 06:26:00,2022-07-21 06:30:00,3,2022-07-06 07:41:00,2022-07-06 07:41:00,2022-07-06 13:21:00
…,…,…,…,…,…,…,…,…
752963510,2,2026-05-16 07:15:00,2026-05-16 08:32:00,2026-05-16 07:15:00,3,2026-05-16 08:32:00,2026-05-16 08:32:00,2026-05-16 13:12:00
753052938,2,2026-05-16 12:12:00,2026-05-16 12:12:00,2026-05-16 12:12:00,3,2026-05-16 12:12:00,2026-05-16 12:12:00,2026-05-16 14:20:28
753371509,2,2026-05-21 05:49:00,2026-05-21 05:49:00,2026-05-21 18:03:23,3,2026-05-21 05:49:00,2026-05-21 05:49:00,2026-05-21 18:03:23


In [ ]:
df_sepsis_v23_maxonset.with_columns(
    (pl.col("max_sepsis_score_v3")-pl.col("max_sepsis_score")).alias('diff_sepsis_score_v3-v2')
).filter(
    pl.col("diff_sepsis_score_v3-v2")<0
)

EncounterEpicCsn,max_sepsis_score,sepsis_instant,min_infect_dt,min_Event_DateTime,max_sepsis_score_v3,sepsis_instant_v3,min_infect_dt_v3,min_Event_DateTime_v3,diff_sepsis_score_v3-v2
i64,i32,datetime[μs],datetime[μs],datetime[μs],i32,datetime[μs],datetime[μs],datetime[μs],i32
663740518,2,2022-07-06 20:56:00,2022-07-08 20:54:36,2022-07-06 20:56:00,1,2022-06-16 09:34:00,2022-06-16 09:37:00,2022-06-16 09:34:00,-1
663941695,2,2022-06-27 06:00:00,2022-06-29 05:28:00,2022-06-27 06:00:00,1,2022-06-21 21:05:00,2022-06-21 21:05:00,2022-06-22 05:00:00,-1
664242978,3,2022-07-15 23:00:00,2022-07-17 16:59:00,2022-07-15 23:00:00,2,2022-06-27 10:03:00,2022-06-27 10:03:00,2022-07-12 05:59:00,-1
664261442,3,2022-07-06 19:16:00,2022-07-07 14:29:00,2022-07-06 19:16:00,2,2022-06-29 21:24:00,2022-06-29 21:24:00,2022-06-30 00:17:00,-1
664529709,2,2022-07-01 14:18:00,2022-07-01 14:18:00,2022-07-01 15:23:00,1,2022-07-01 14:10:00,2022-07-01 14:18:00,2022-07-01 14:10:00,-1
…,…,…,…,…,…,…,…,…,…
749212782,3,2026-03-25 15:07:00,2026-03-25 15:07:00,2026-03-27 10:00:00,2,2026-03-24 23:26:00,2026-03-25 01:30:00,2026-03-24 23:26:00,-1
749957663,3,2026-04-03 20:12:00,2026-04-04 16:01:44,2026-04-03 20:12:00,1,2026-04-25 20:44:00,2026-04-26 16:04:00,2026-04-25 20:44:00,-2
751015083,3,2026-04-17 23:16:00,2026-04-18 23:51:00,2026-04-17 23:16:00,2,2026-04-21 14:00:00,2026-04-23 13:33:00,2026-04-21 14:00:00,-1


In [ ]:
# Focusing only on the encounter categorized by both v2 and v3 in the maxonset
df_sepsis_v23_firstonset = df_sepsis_v2_firstonset.join(
    df_sepsis_v3_firstonset, on=['EncounterEpicCsn'], how='inner', suffix='_v3'
)
# df_sepsis_v23_firstonset
df_sepsis_v23_firstonset.filter(
    pl.col("sepsis_score_v3")!=pl.col("sepsis_score")
).sort(by=['EncounterEpicCsn'])

df_sepsis_v23_firstonset.with_columns(
    (pl.col("sepsis_score_v3")-pl.col("sepsis_score")).alias('diff_sepsis_score_v3-v2')
)['diff_sepsis_score_v3-v2'].value_counts(sort=True)

df_sepsis_v23_firstonset.with_columns(
    (pl.col("sepsis_score_v3")-pl.col("sepsis_score")).alias('diff_sepsis_score_v3-v2')
).filter(
	pl.col('diff_sepsis_score_v3-v2')<0
)

enc_shock_v2_and_noshock_v3 = df_sepsis_v23_firstonset.filter(
    (pl.col("sepsis_score_v3")<3) & (pl.col("sepsis_score")==3)
)['EncounterEpicCsn'].unique()

In [ ]:
df_sepsis_v23_maxonset.filter(
    pl.col("EncounterEpicCsn").is_in([662715699, 662358356])
)

EncounterEpicCsn,max_sepsis_score,sepsis_instant,min_infect_dt,min_Event_DateTime,max_sepsis_score_v3,sepsis_instant_v3,min_infect_dt_v3,min_Event_DateTime_v3
i64,i32,datetime[μs],datetime[μs],datetime[μs],i32,datetime[μs],datetime[μs],datetime[μs]
662358356,3,2022-06-13 04:59:00,2022-06-13 04:59:00,2022-06-13 05:15:00,3,2022-06-13 04:59:00,2022-06-13 04:59:00,2022-06-13 05:21:00
662715699,3,2022-06-29 11:02:00,2022-06-30 20:59:00,2022-06-29 11:02:00,3,2022-07-01 22:00:00,2022-07-03 10:47:00,2022-07-01 22:00:00


In [ ]:
df_all_v3.filter(
    pl.col("EncounterEpicCsn").is_in([662715699])&
	(pl.col("Event_DateTime") == (dt.datetime(2022, 7, 1, 11)))&
	# (pl.col("Event_DateTime") >= (dt.datetime(2022, 7, 1, 11)-dt.timedelta(hours=48)))&
	# (pl.col("Event_DateTime") <= (dt.datetime(2022, 7, 1, 11)+dt.timedelta(hours=48)))&
	# (pl.col("Event_Grouper").str.to_lowercase().str.starts_with('Glasgow Coma Score'))
	(pl.col("Event_Grouper") == 'Glasgow Coma Score')
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
i64,datetime[μs],str,str,str,f64,str,str,i64,f64,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
662715699,2022-07-01 11:00:00,"""Flowsheet""","""Glasgow Coma Score""","""CPM S25 R AS SC SUM SCORE""",12.0,"""12""",null,94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""


In [ ]:
df_o2vent_3.filter(
    pl.col("EncounterEpicCsn").is_in([662715699])&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 7, 1, 11)-dt.timedelta(hours=48)))&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 7, 1, 11)+dt.timedelta(hours=48)))&
	(pl.col("pulmonary_dysfunction_flag")==1)
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,pf_ratio,PF_Ratio_Flag,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],str,str,str,f64,str,str,i64,f64,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,f64,i32,i32,i32,i32,i32,i32
662715699,2022-06-30 21:05:00,"""Lab Results""","""PAO2""","""PO2 ART""",68.0,"""68""","""1""",94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",188.888889,1,null,null,null,null,1
662715699,2022-07-02 08:00:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",null,null,1,null,null,null,1
662715699,2022-07-02 16:16:00,"""Lab Results""","""PAO2""","""PO2 ART""",87.0,"""87""",null,94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",174.0,1,null,null,null,null,1
662715699,2022-07-02 02:18:00,"""Lab Results""","""PAO2""","""PO2 ART""",47.0,"""47""","""1""",94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:0

In [ ]:
df_all_v3.filter(
    pl.col("EncounterEpicCsn").is_in([662715699])&
	# (pl.col("Event_DateTime") >= (dt.datetime(2022, 7, 1, 11)-dt.timedelta(hours=48)))&
	# (pl.col("Event_DateTime") <= (dt.datetime(2022, 7, 1, 11)+dt.timedelta(hours=48)))&
	(pl.col("Event_Grouper") == 'Lactate')
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
i64,datetime[μs],str,str,str,f64,str,str,i64,f64,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
662715699,2022-06-29 11:02:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",1.8,"""1.8""",null,94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""
662715699,2022-06-29 12:15:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",2.6,"""2.6""","""1""",94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""
662715699,2022-06-29 13:12:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",2.9,"""2.9""","""1""",94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""
662715699,2022-06-29 13:54:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",3.1,"""3.1""","""1""",94142911,75.641341,"""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic h

In [ ]:
df_organdysf_3.filter(
    pl.col("EncounterEpicCsn").is_in([662715699])&
	(pl.col("Event_DateTime") == (dt.datetime(2022, 7, 1, 11)))&
	# (pl.col("Event_DateTime") >= (dt.datetime(2022, 7, 1, 11)-dt.timedelta(hours=48)))&
	# (pl.col("Event_DateTime") <= (dt.datetime(2022, 7, 1, 11)+dt.timedelta(hours=48)))&
    (pl.col("organ_dysfunction_total")>0) 
)

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,Baseline_Bilirubin,inr_criteria_flag,Baseline_PulseRate,Baseline_RespiratoryRate,Baseline_WBC,aptt_criteria_flag,coagulation_failure_flag,Baseline_SBP,platelets50_criteria_flag,Baseline_eGFR,Baseline_Platelets,Baseline_Creatinine,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total
i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,f64,f64,f64,i32,i32,f64,i32,f64,f64,f64,i32,i32,i32,i64,i64
662715699,2022-07-01 11:00:00,99.9,89.0,17.0,null,null,22.03,1.38,40.0,null,null,130.0,null,null,12.0,null,null,null,0.02,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,1,1


In [ ]:
df_shock_3.filter(
    pl.col("EncounterEpicCsn").is_in([662715699])&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 7, 1, 22, 2)-dt.timedelta(hours=48)))&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 7, 1, 22, 2)+dt.timedelta(hours=48)))
).sort(by=['Event_DateTime'])

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,Baseline_Bilirubin,inr_criteria_flag,Baseline_PulseRate,Baseline_RespiratoryRate,Baseline_WBC,aptt_criteria_flag,coagulation_failure_flag,Baseline_SBP,platelets50_criteria_flag,Baseline_eGFR,Baseline_Platelets,Baseline_Creatinine,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total,sbp90_flag,sbpdelta40_flag,map65_flag,lactate4_flag,vasopressor_flag,septic_shock_flag
i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,f64,f64,f64,i32,i32,f64,i32,f64,f64,f64,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64,i64
662715699,2022-06-29 23:00:00,97.3,66.0,18.0,133.0,116.3,15.09,0.91,66.0,null,null,128.0,1.3,null,null,null,null,0.03,0.02,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,null,1,0
662715699,2022-06-29 23:05:00,97.3,66.0,18.0,133.0,116.3,15.09,0.91,66.0,null,null,128.0,1.3,null,null,null,null,0.03,0.02,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,null,1,0
662715699,2022-06-29 23:26:35,97.3,66.0,18.0,133.0,116.3,15.09,0.91,66.0,null,null,128.0,1.3,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,null,1,0
662715699,2022-06-30 00:00:00,97.3,68.0,18.0,127.0,92.3,15.09,0.91,66.0,null,null,128.0,1.3,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,null,1,0
662715699,2022-06-30 01:00:00,97.3,68.0,18.0,127.0,92.3,15.09,0.91,66.0,null,null,128.0,1.3,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
662715699,2022-07-03 20:36:00,100.4,58.0,20.0,136.0,86.0,19.79,1.43,38.0,null,null,144.0,null,null,11.0,null,null,null,0.03,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,1,1,0,0,0,null,1,0
662715699,2022-07-03 20:40:00,100.4,58.0,20.0,136.0,86.0,19.79,1.43,38.0,null,null,144.0,null,null,11.0,null,null,null,0.05,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,1,1,0,0,0,null,1,0
662715699,2022-07-03 21:00:00,100.4,61.0,21.0,155.0,95.7,19.79,1.43,38.0,null,null,144.0,null,null,11.0,null,null,null,0.05,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,1,1,0,0,0,null,1,0


In [ ]:
df_sirs_2

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,WBC_Abnormal_Flag,Temp_Abnormal_Flag,Resp_Rate_High_Flag,Pule_High_Flag,sirs_score
i64,datetime[μs],str,str,str,f64,str,str,i64,str,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32
662715699,2022-06-02 01:53:06.913,"""Diagnosis Event""","""Encounter Diagnosis""","""3-vessel CAD""",null,"""I25.10""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,0,0,0
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Pulse""","""PULSE""",72.0,"""72""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,0,0,0
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Respirations""","""RESPIRATIONS""",18.0,"""18""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,1,0,1
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Blood Pressure""","""BLOOD PRESSURE""",null,"""176/59""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""

In [ ]:
df_organdysf_2.filter(
	(pl.col("EncounterEpicCsn") == 662715699)&
	(pl.col("Event_DateTime") = (dt.datetime(2022, 6, 29, 11, 2)))
)

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,vent_status_time,vent_status_flag,Baseline_Creatinine,Baseline_eGFR,Baseline_Bilirubin,Baseline_Platelets,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_WBC,cardiovascular_failure_flag,coagulation_failure_flag,aptt_criteria_flag,platelets100_criteria_flag,platelets50_criteria_flag,inr_criteria_flag,pulmonary_failure_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total
i64,datetime[μs],f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i64,i32,i32,i64,i64
662715699,2022-06-29 11:02:00,97.6,72.0,18.0,176,null,null,null,null,null,2.2,null,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 12:10:00,97.6,72.0,18.0,176,null,null,null,null,null,1.8,null,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,0,0,0,0,0
662715699,2022-06-29 12:15:00,97.6,72.0,18.0,176,null,7.77,null,null,null,1.8,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,0,0,0,0,0
662715699,2022-06-29 13:12:00,97.6,72.0,18.0,176,null,7.77,null,null,null,2.6,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 13:54:00,97.6,72.0,18.0,176,null,7.77,null,null,null,2.9,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
662715699,2022-07-25 01:46:05.110,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2022-07-12 20:55:00,"""O2 Delivery High-Flow""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,1,0,0,0,1
662715699,2022-07-26 02:38:26.133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2022-07-12 20:55:00,"""O2 Delivery High-Flow""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,1,0,0,0,1
662715699,2022-07-26 02:38:26.133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2022-07-14 10:20:00,"""Vent on Documentation""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,1,0,0,0,1


In [42]:
df_sirs_2.filter(
	(pl.col("EncounterEpicCsn") == 662715699)&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 6, 29, 11, 2)))&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 6, 29, 11, 2))-dt.timedelta(hours=6))&
	(pl.col("Event_Grouper")=='Lactate')
	# pl.col()
	# (pl.col("Event_DateTime") >= (dt.datetime(2022, 6, 29, 11, 2)-dt.timedelta(hours=48)))&
	# (pl.col("Event_DateTime") <= (dt.datetime(2022, 6, 29, 11, 2)+dt.timedelta(hours=48)))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,WBC_Abnormal_Flag,Temp_Abnormal_Flag,Resp_Rate_High_Flag,Pule_High_Flag,sirs_score
i64,datetime[μs],str,str,str,f64,str,str,i64,str,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32
662715699,2022-06-29 07:59:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",0.9,"""0.9""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,1,0,1
662715699,2022-06-29 09:21:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",1.0,"""1.0""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,1,0,1
662715699,2022-06-29 10:23:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",2.2,"""2.2""","""1""",94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,1,0,1
662715699,2022-06-29 11:02:00,"""Lab Results""","""Lactate""","""LACTIC ACID (ISTAT)""",1.8,"""1.8""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery"""

In [44]:
df_sirs_3.filter(
    	(pl.col("EncounterEpicCsn") == 662715699)&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 6, 29, 11, 2)))&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 6, 29, 11, 2))-dt.timedelta(hours=6))
	# (pl.col("Event_Grouper")=='Lactate')
)

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,Temp_Abnormal_Flag,Pule_High_Flag,Resp_Rate_High_Flag,WBC_Abnormal_Flag,sirs_score
i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32
662715699,2022-06-29 09:21:00,97.6,72.0,18.0,176.0,98.0,null,null,null,null,1.0,null,null,null,null,null,null,null,null,0,0,0,0,0
662715699,2022-06-29 11:02:00,97.6,72.0,18.0,176.0,98.0,null,null,null,null,1.8,null,null,null,null,null,null,null,null,0,0,0,0,0
662715699,2022-06-29 10:23:00,97.6,72.0,18.0,176.0,98.0,null,null,null,null,2.2,null,null,null,null,null,null,null,null,0,0,0,0,0
662715699,2022-06-29 07:59:00,97.6,72.0,18.0,176.0,98.0,null,null,null,null,0.9,null,null,null,null,null,null,null,null,0,0,0,0,0
662715699,2022-06-29 06:14:00,97.6,72.0,18.0,176.0,98.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0


In [31]:
df_organdysf_2.filter(
	(pl.col("EncounterEpicCsn") == 662715699)&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 6, 29, 11, 2)-dt.timedelta(hours=48)))&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 6, 29, 11, 2)+dt.timedelta(hours=48)))&
	(pl.col("organ_dysfunction_total")>0)
).sort(by=['Event_DateTime'])

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,vent_status_time,vent_status_flag,Baseline_Creatinine,Baseline_eGFR,Baseline_Bilirubin,Baseline_Platelets,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_WBC,cardiovascular_failure_flag,coagulation_failure_flag,aptt_criteria_flag,platelets100_criteria_flag,platelets50_criteria_flag,inr_criteria_flag,pulmonary_failure_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total
i64,datetime[μs],f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i64,i32,i32,i64,i64
662715699,2022-06-29 11:02:00,97.6,72.0,18.0,176,null,null,null,null,null,2.2,null,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 13:12:00,97.6,72.0,18.0,176,null,7.77,null,null,null,2.6,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 13:54:00,97.6,72.0,18.0,176,null,7.77,null,null,null,2.9,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 14:51:00,null,null,null,null,null,7.77,null,null,null,3.1,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
662715699,2022-06-29 14:55:00,96.3,89.0,18.0,null,84.0,7.77,null,null,null,3.1,136.0,null,null,null,null,null,null,null,null,null,0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,1,0,0,0,0,0,0,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
662715699,2022-07-01 08:00:00,99.5,88.0,17.0,null,73.0,22.03,1.38,40.0,null,null,130.0,1.1,null,12.0,null,null,null,0.02,2022-06-30 14:09:00,"""Vent off Documentation""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,0,0,0,1,1
662715699,2022-07-01 08:22:00,99.5,87.0,17.0,null,68.0,22.03,1.38,40.0,null,null,130.0,1.1,null,12.0,null,null,null,0.02,2022-06-30 14:09:00,"""Vent off Documentation""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,0,0,0,1,1
662715699,2022-07-01 09:00:00,99.5,87.0,17.0,null,68.0,22.03,1.38,40.0,null,null,130.0,1.1,null,12.0,null,null,null,0.02,2022-06-30 14:09:00,"""Vent off Documentation""",0.845,69.5,0.266667,253.666667,138.0,16.0,63.0,9.943333,0,0,0,0,0,0,0,0,0,1,1


In [28]:
df_shock_3.filter(
	(pl.col("EncounterEpicCsn") == 662715699)&
	(pl.col("Event_DateTime") >= (dt.datetime(2022, 6, 29, 11, 2)-dt.timedelta(hours=48)))&
	(pl.col("Event_DateTime") <= (dt.datetime(2022, 6, 29, 11, 2)+dt.timedelta(hours=48)))&
	(pl.col("septic_shock_flag")==1)
)

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,Baseline_Bilirubin,inr_criteria_flag,Baseline_PulseRate,Baseline_RespiratoryRate,Baseline_WBC,aptt_criteria_flag,coagulation_failure_flag,Baseline_SBP,platelets50_criteria_flag,Baseline_eGFR,Baseline_Platelets,Baseline_Creatinine,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total,sbp90_flag,sbpdelta40_flag,map65_flag,lactate4_flag,vasopressor_flag,septic_shock_flag
i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,f64,f64,f64,i32,i32,f64,i32,f64,f64,f64,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64,i64
662715699,2022-06-29 15:55:00,96.6,85.0,18.0,78.0,64.0,15.09,0.91,66.0,null,3.1,128.0,1.3,null,null,null,null,0.03,0.02,1,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,1,0,1,1,0,1,1
662715699,2022-06-29 16:00:00,96.6,86.0,8.0,82.0,62.0,15.09,0.91,66.0,null,3.1,128.0,1.3,null,null,null,null,0.03,0.02,1,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,1,0,1,1,0,1,1
662715699,2022-06-29 16:05:00,96.6,86.0,18.0,91.0,69.0,15.09,0.91,66.0,null,3.1,128.0,1.3,null,null,null,null,0.03,0.02,1,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,1,0,1,0,0,1,1
662715699,2022-06-29 16:08:00,96.6,86.0,18.0,91.0,69.0,15.09,0.91,66.0,null,3.4,128.0,1.3,null,null,null,null,0.03,0.02,1,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,1,0,1,0,0,1,1
662715699,2022-06-29 16:10:00,96.6,88.0,18.0,92.0,71.3,15.09,0.91,66.0,null,3.4,128.0,1.3,null,null,null,null,0.03,0.02,1,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,1,0,1,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
662715699,2022-06-30 07:00:00,98.4,69.0,12.0,115.0,66.3,16.82,1.28,44.0,0.3,1.9,152.0,null,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,0,1,1
662715699,2022-06-30 07:40:00,98.4,69.0,12.0,115.0,66.3,16.82,1.28,44.0,0.3,1.9,152.0,null,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,0,1,1
662715699,2022-06-30 08:00:00,98.2,69.0,12.0,118.0,72.0,16.82,1.28,44.0,0.3,1.9,152.0,null,null,null,null,null,0.03,0.08,0,0.266667,0,63.0,16.0,9.943333,0,0,138.0,0,69.5,253.666667,0.845,0,0,0,0,0,0,0,0,0,1,1


In [19]:
df_sirs_2_enc = df_sirs_2.filter(	
	pl.col("EncounterEpicCsn") == 662715699
)
df_sirs_2_enc

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,WBC_Abnormal_Flag,Temp_Abnormal_Flag,Resp_Rate_High_Flag,Pule_High_Flag,sirs_score
i64,datetime[μs],str,str,str,f64,str,str,i64,str,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32
662715699,2022-06-02 01:53:06.913,"""Diagnosis Event""","""Encounter Diagnosis""","""3-vessel CAD""",null,"""I25.10""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,0,0,0
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Pulse""","""PULSE""",72.0,"""72""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,0,0,0
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Respirations""","""RESPIRATIONS""",18.0,"""18""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""22""","""UH 09B CVICU""","""UH 09B CVICU""","""Elective""","""Home & Outside Location""","""OR Admission""","""3-vessel CAD""","""Atherosclerotic heart disease …","""Dilation of Coronary Artery, O…",1,"""NPOA-3""","""NPOA-3""",138.0,16.0,63.0,0.845,253.666667,0.266667,69.5,9.943333,"""2022-07-03 10:47:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",0,0,1,0,1
662715699,2022-06-29 06:14:00,"""Flowsheet""","""Blood Pressure""","""BLOOD PRESSURE""",null,"""176/59""",null,94142911,"""75.641341""","""Female""","""Non-Hispanic/Latino""","""White""","""0""",662715699,2022-06-29 00:00:00,2022-07-21 00:00:00,2022-06-29 05:59:00,2022-06-30 08:03:00,2022-06-29 05:59:00,"""No""","""Inpatient""","""Inpatient""","""Cardiac Surgery""","""

### Combine Organ dysfunctino and Pulmonary dysfunction into one dataframe

In [17]:
df_o2vent_3_cols = ['EncounterEpicCsn', "Event_DateTime"] + [c for c in df_o2vent_3.columns if c.endswith('_flag') or c.startswith('last_') or c.endswith('_score') or c.endswith('_ratio')]
df_o2vent_3_cols

['EncounterEpicCsn',
 'Event_DateTime',
 'pf_ratio',
 'vent_onoff_flag',
 'vent_doc_flag',
 'o2_vent_flag',
 'vent_end_flag',
 'pulmonary_dysfunction_flag']

In [18]:
df_o2vent_3_refined = df_o2vent_3.filter(
	# Remove rows where all the following columns are nulls
    # ~pl.all_horizontal(
    #     pl.col([
    #         'pf_ratio',	'PF_Ratio_Flag', 'vent_onoff_flag',	'vent_doc_flag', 'o2_vent_flag', 'vent_end_flag', 'pulmonary_dysfunction_flag'
	# 	]).is_null()
	# )

	pl.col("pulmonary_dysfunction_flag")>0

).select(df_o2vent_3_cols).unique(subset=['EncounterEpicCsn', 'Event_DateTime'])

In [20]:
df_organdysf_3_refined =df_organdysf_3.filter(
    pl.col("organ_dysfunction_total")>0
).select(
	['EncounterEpicCsn', 'Event_DateTime']+[c for c in df_organdysf_3.columns if c.endswith('_flag') or c.startswith('last_') or c.endswith('_score') or c.endswith('_ratio')]+['organ_dysfunction_total']
).unique(subset=['EncounterEpicCsn', 'Event_DateTime'])

In [21]:
df_organdysfunction_backbone = pl.concat([
    df_organdysf_3_refined.select("EncounterEpicCsn", "Event_DateTime", pl.lit("organdys").alias("organdysfunction_type")),
    df_o2vent_3_refined.select("EncounterEpicCsn", "Event_DateTime", pl.lit("pulmonary").alias("organdysfunction_type"))
], how='vertical')

In [22]:
df_organ_pulmonary_3 = df_organdysfunction_backbone.join(
    df_organdysf_3_refined, on=['EncounterEpicCsn', 'Event_DateTime'], how='left'
).join(df_o2vent_3_refined, on=['EncounterEpicCsn', 'Event_DateTime'], how='left').sort(by=['EncounterEpicCsn', 'Event_DateTime'])

### Combine flag dataframes with sepsis dataframes to map the cause v3

In [23]:
df_organ_pulmonary_3

EncounterEpicCsn,Event_DateTime,organdysfunction_type,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,inr_criteria_flag,aptt_criteria_flag,coagulation_failure_flag,platelets50_criteria_flag,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total,pf_ratio,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,f64,i32,i32,i32,i32,i32
659308243,2022-06-14 16:40:00,"""organdys""",98.2,76.0,18.0,138.0,95.3,null,null,null,null,2.1,null,null,null,null,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
659308243,2022-06-14 17:52:00,"""organdys""",98.2,76.0,18.0,138.0,95.3,null,null,null,null,2.5,null,null,null,null,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
659308243,2022-06-14 18:48:00,"""organdys""",98.2,76.0,18.0,138.0,95.3,null,null,null,null,3.1,null,null,null,null,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
659308243,2022-06-14 21:35:00,"""organdys""",null,null,null,null,null,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
659308243,2022-06-14 21:49:00,"""organdys""",null,null,null,null,null,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753579730,2026-05-26 09:01:00,"""organdys""",98.8,111.0,26.0,125.0,91.0,20.31,0.79,100.0,0.3,3.6,363.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
753579730,2026-05-26 10:00:00,"""organdys""",98.8,113.0,24.0,127.0,94.3,20.31,0.79,100.0,0.3,3.6,363.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
753579730,2026-05-26 10:13:00,"""organdys""",98.8,113.0,24.0,127.0,94.3,20.31,0.79,100.0,0.3,3.6,363.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null


#### Combining septic shock flags with sepsis dataframe

In [24]:
df_sepsis_3_with_shock = df_sepsis_v3.join(
    df_shock_3, on=['EncounterEpicCsn', 'Event_DateTime'], how='left'
).filter(
    pl.col('sepsis_score') == 3
)

df_sepsis_3_with_shock = df_sepsis_3_with_shock.drop([c for c in df_sepsis_3_with_shock.columns if c.endswith('_right')])
# df_sepsis_3_with_shock = df_sepsis_3_with_shock.join(
#     df_o2vent_3.filter(
#     pl.col("pulmonary_dysfunction_flag") == 1
# 	).select(["EncounterEpicCsn", 'Event_DateTime', 'pf_ratio']+[c for c in df_o2vent_3.columns if c.endswith('_flag')]),
#     on='EncounterEpicCsn', how='left'
# )

# df_sepsis_3_with_shock = df_sepsis_3_with_shock.join(
#     df_organ_pulmonary_3.filter(
#     	(pl.col("pulmonary_dysfunction_flag") == 1)|(pl.col("organ_dysfunction_total")>0)
# 	).select(["EncounterEpicCsn", 'Event_DateTime', 'pf_ratio']+[c for c in df_o2vent_3.columns if c.endswith('_flag')]),
#     on='EncounterEpicCsn', how='left'
# )

df_shock_orgrandysfunction_3 = df_sepsis_3_with_shock.join_asof(
	df_organ_pulmonary_3,
	left_on='sepsis_instance',
	right_on='Event_DateTime',
	by='EncounterEpicCsn',
	strategy='nearest',
    suffix='_organdysfunction'
)

/tmp/ipykernel_1784179/789479857.py:22: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_shock_orgrandysfunction_3 = df_sepsis_3_with_shock.join_asof(


In [25]:
df_shock_orgrandysfunction_3

EncounterEpicCsn,infect_dt,Event_DateTime,sepsis_instance,sepsis_score,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,Baseline_Bilirubin,inr_criteria_flag,Baseline_PulseRate,Baseline_RespiratoryRate,Baseline_WBC,aptt_criteria_flag,coagulation_failure_flag,Baseline_SBP,platelets50_criteria_flag,Baseline_eGFR,Baseline_Platelets,Baseline_Creatinine,platelets100_criteria_flag,…,septic_shock_flag,Event_DateTime_organdysfunction,organdysfunction_type,last_temp_8h_organdysfunction,last_pulse_8h_organdysfunction,last_resp_8h_organdysfunction,last_sbp_8h_organdysfunction,last_map_8h_organdysfunction,last_wbc_12h_organdysfunction,last_creatinine_12h_organdysfunction,last_egfr_12h_organdysfunction,last_bilirubin_12h_organdysfunction,last_lactate_6h_organdysfunction,last_platelets_24h_organdysfunction,last_inr_12h_organdysfunction,last_aptt_24h_organdysfunction,last_gcs_12h_organdysfunction,last_vasopressin_24h_organdysfunction,last_phenylephrine_24h_organdysfunction,last_norepinephrine_24h_organdysfunction,last_epinephrine_24h_organdysfunction,cardiovascular_failure_flag_organdysfunction,inr_criteria_flag_organdysfunction,aptt_criteria_flag_organdysfunction,coagulation_failure_flag_organdysfunction,platelets50_criteria_flag_organdysfunction,platelets100_criteria_flag_organdysfunction,renal_failure_flag_organdysfunction,hepatic_failure_flag_organdysfunction,neurological_failure_flag_organdysfunction,organ_dysfunction_total_organdysfunction,pf_ratio,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],datetime[μs],datetime[μs],i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,f64,f64,f64,i32,i32,f64,i32,f64,f64,f64,i32,…,i64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,f64,i32,i32,i32,i32,i32
659308243,2022-06-14 22:10:00,2022-06-16 03:33:00,2022-06-14 22:10:00,3,98.7,77.0,23.0,111.0,73.0,16.99,2.95,22.0,null,2.0,130.0,null,null,3.0,0.04,null,0.22,null,0,0.96,0,63.0,16.0,5.842,0,0,134.0,0,60.333333,137.8,1.24,0,…,1,2022-06-14 22:10:00,"""organdys""",96.8,106.0,27.0,144.0,100.0,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
659308243,2022-06-14 22:10:00,2022-06-16 02:50:27.320,2022-06-14 22:10:00,3,98.7,64.0,25.0,101.0,63.7,16.99,2.95,22.0,null,2.0,130.0,null,null,3.0,0.04,null,0.22,null,0,0.96,0,63.0,16.0,5.842,0,0,134.0,0,60.333333,137.8,1.24,0,…,1,2022-06-14 22:10:00,"""organdys""",96.8,106.0,27.0,144.0,100.0,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
659308243,2022-06-14 22:10:00,2022-06-15 01:45:00,2022-06-14 22:10:00,3,96.8,106.0,16.0,134.0,77.3,7.17,2.01,35.0,null,5.3,176.0,null,null,9.0,null,null,0.01,null,1,0.96,0,63.0,16.0,5.842,0,0,134.0,0,60.333333,137.8,1.24,0,…,1,2022-06-14 22:10:00,"""organdys""",96.8,106.0,27.0,144.0,100.0,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
659308243,2022-06-14 22:10:00,2022-06-15 02:00:00,2022-06-14 22:10:00,3,96.8,103.0,17.0,99.0,61.0,7.17,2.01,35.0,null,5.3,176.0,null,null,9.0,null,null,0.01,null,1,0.96,0,63.0,16.0,5.842,0,0,134.0,0,60.333333,137.8,1.24,0,…,1,2022-06-14 22:10:00,"""organdys""",96.8,106.0,27.0,144.0,100.0,null,null,null,null,3.1,null,null,null,9.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
659308243,2022-06-14 22:10:00,2022-06-15 02:03:00,2022-06-14 22:10:00,3,96.8,103.0,17.0,99.0,61.0,7.17,2.01,35.0,null,5.3,176.0,null,null,9.0,null,null,0.01,null,1,0.96,0,63.0,16.0,5.842,0,0,134.0,0,60.333333,137.8,1.24,0,…,1,2022-

In [30]:
df_sepsis_3_v3_all_comb = df_shock_orgrandysfunction_3.join(
    df_shock_orgrandysfunction_3.join(
		df_shock_orgrandysfunction_3.group_by(
			"EncounterEpicCsn"
		).agg(
			pl.col("sepsis_score").max().alias('max_sepsis_score')
		), on="EncounterEpicCsn" , how='inner'
	).filter(
		pl.col('max_sepsis_score') == pl.col('sepsis_score')
	).group_by(
		"EncounterEpicCsn"
	).agg(
		pl.col("sepsis_instance").min(),
		pl.col("max_sepsis_score").first()
	), on=['EncounterEpicCsn', 'sepsis_instance'], how='inner'
)

In [34]:
df_sepsis_3_v3_all_comb.select(
	['EncounterEpicCsn', 'infect_dt', 'Event_DateTime', pl.col('sepsis_instance').alias("sepsis_instant"), 'sepsis_score']
	+
	[c for c in df_shock_orgrandysfunction_3.columns if c.endswith('_flag') or c.endswith('_ratio') or c.endswith('_total')]
).write_csv('./sepsis_3_v3_all_comb.csv')

#### Combining organdysfunc flags with sepsis 2 

In [35]:
df_sepsis_2_with_organdysf = df_sepsis_v2.join(
    df_organ_pulmonary_3, on=['EncounterEpicCsn', 'Event_DateTime'], how='left'
).filter(
    pl.col('sepsis_score') == 2
)
df_sepsis_2_with_organdysf

EncounterEpicCsn,infect_dt,Event_DateTime,sepsis_instance,sepsis_score,organdysfunction_type,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,inr_criteria_flag,aptt_criteria_flag,coagulation_failure_flag,platelets50_criteria_flag,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total,pf_ratio,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],datetime[μs],datetime[μs],i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,f64,i32,i32,i32,i32,i32
659308243,2022-06-14 22:10:00,2022-06-14 17:52:00,2022-06-14 17:52:00,2,"""organdys""",98.2,76.0,18.0,138.0,95.3,null,null,null,null,2.5,null,null,null,null,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
660956097,2022-06-28 06:10:00,2022-06-28 01:09:00,2022-06-28 01:09:00,2,"""organdys""",97.7,61.0,18.0,132.0,96.7,18.96,0.92,88.0,null,3.3,211.0,null,null,14.0,null,null,null,null,1,0,0,0,0,0,0,0,1,2,null,null,null,null,null,null
661424247,2022-08-28 05:30:00,2022-08-28 02:18:00,2022-08-28 02:18:00,2,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
661471289,2022-06-07 19:47:00,2022-06-07 12:07:00,2022-06-07 12:07:00,2,"""organdys""",97.9,67.0,28.0,92.0,62.7,5.22,1.24,71.0,null,null,317.0,1.1,null,14.0,null,null,null,null,0,0,0,0,0,0,0,0,1,1,null,null,null,null,null,null
662152913,2022-08-15 16:45:00,2022-08-15 17:00:00,2022-08-15 16:45:00,2,"""organdys""",96.8,125.0,22.0,90.0,null,6.61,null,null,null,4.5,117.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753545238,2026-05-23 16:06:00,2026-05-23 11:51:00,2026-05-23 11:51:00,2,"""organdys""",100.0,119.0,35.0,187.0,114.3,1.54,0.7,98.0,8.0,6.5,141.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,1,0,2,null,null,null,null,null,null
753546980,2026-05-23 14:11:00,2026-05-23 15:32:00,2026-05-23 14:11:00,2,"""organdys""",99.0,121.0,16.0,120.0,84.0,23.83,1.28,59.0,0.9,5.3,354.0,null,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null
753558469,2026-05-24 18:39:00,2026-05-24 18:54:00,2026-05-24 18:39:00,2,"""organdys""",97.7,144.0,48.0,115.0,89.0,12.39,0.82,81.0,1.6,4.5,284.0,1.4,null,15.0,null,null,null,null,1,0,0,0,0,0,0,0,0,1,null,null,null,null,null,null


In [46]:
df_o2vent_3.drop_nulls(subset=[
    'pf_ratio', 'vent_onoff_flag', 'vent_doc_flag',	'o2_vent_flag',	'vent_end_flag', 'pulmonary_dysfunction_flag'
])

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,pf_ratio,PF_Ratio_Flag,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],str,str,str,f64,str,str,i64,f64,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,f64,i32,i32,i32,i32,i32,i32


In [47]:
df_o2vent_3.select(
    ['EncounterEpicCsn', 'Event_DateTime']+[c for c in df_o2vent_3.columns if c.endswith('_flag') or c.startswith('last_') or c.endswith('_score') or c.endswith('_ratio')]
)

EncounterEpicCsn,Event_DateTime,pf_ratio,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],f64,i32,i32,i32,i32,i32
670101787,2022-10-13 22:10:00,null,null,null,null,null,null
692424715,2023-11-13 10:45:00,null,null,null,null,null,null
686550527,2023-08-08 03:52:00,null,null,null,null,null,null
674101030,2023-01-08 20:45:00,null,null,null,null,null,null
690415709,2023-10-06 19:53:00,null,null,null,null,null,null
…,…,…,…,…,…,…,…
718191846,2024-12-22 07:52:00,null,null,null,null,null,null
695767263,2024-02-04 04:00:00,null,null,null,null,null,null
746782688,2026-04-01 13:50:00,null,null,null,null,null,null


In [48]:
df_organdysf_3.select(
    ['EncounterEpicCsn', 'Event_DateTime']+[c for c in df_organdysf_3.columns if c.endswith('_flag') or c.startswith('last_') or c.endswith('_score') or c.endswith('_ratio')]
)

EncounterEpicCsn,Event_DateTime,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,inr_criteria_flag,aptt_criteria_flag,coagulation_failure_flag,platelets50_criteria_flag,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag
i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64
659308243,2022-06-14 10:50:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
659308243,2022-06-14 11:22:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
659308243,2022-06-14 11:23:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
659308243,2022-06-14 11:52:00,98.2,76.0,18.0,138.0,95.3,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
659308243,2022-06-14 13:21:00,98.2,76.0,18.0,138.0,95.3,null,null,null,null,1.6,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753774459,2026-05-28 15:00:00,null,null,null,132.0,99.3,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
753774459,2026-05-28 19:27:00,null,null,null,149.0,103.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0
753774459,2026-05-28 22:28:00,null,null,null,141.0,107.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0


In [49]:
df_o2vent_3

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN,pf_ratio,PF_Ratio_Flag,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],str,str,str,f64,str,str,i64,f64,str,str,str,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,f64,i32,i32,i32,i32,i32,i32
670101787,2022-10-13 22:10:00,"""Flowsheet""","""Pulse""","""PULSE""",83.0,"""83""",null,92717954,95.216974,"""Male""","""Unknown""","""White""","""0""",670101787,2022-10-11 00:00:00,2022-11-08 00:00:00,2022-10-11 10:23:00,2022-10-11 15:42:00,2022-10-11 10:23:00,"""No""","""Inpatient""","""Observation""","""ENT""","""28""","""UH 07O""","""UH 06G""","""Elective""","""Physician Ofc/OP Clinic""","""OR Admission""","""Dysphagia, unspecified type""","""Malignant neoplasm of mandible…","""Resection of Left Mandible, Op…",0,"""NPOA-1""","""NPOA-1""",120.0,16.0,76.0,0.835,310.0,0.4,80.0,6.41,"""2022-10-21 05:41:28.0000000""","""IV Antibiotics + Blood Culture…","""1""","""0""","""0""","""0""","""0""","""0""",null,null,null,null,null,null,null
692424715,2023-11-13 10:45:00,"""Flowsheet""","""Arterial Blood Pressure Mean""","""UTSW R ARTERIAL BLOOD PRESSURE…",75.0,"""75""",null,71157677,83.805612,"""Female""","""Hispanic or Latino""","""White""","""0""",692424715,2023-11-03 00:00:00,2023-11-22 00:00:00,2023-11-03 14:18:00,2023-11-03 16:15:00,2023-11-03 16:59:00,"""Yes""","""Inpatient""","""Inpatient""","""Hospital Medicine""","""19""","""UH 08B MICU""","""UH 09O""","""Emergency""","""Skilled Nursing, Intermediate …","""ED Admission""","""Acute confusional state""","""Sepsis, unspecified organism""","""Respiratory Ventilation, Great…",0,"""POA-3""","""POA-3""",132.0,null,null,null,196.947368,null,21.953571,8.178947,"""2023-11-03 14:25:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0""",null,null,null,null,null,null,null
686550527,2023-08-08 03:52:00,"""Flowsheet""","""Weight""","""WEIGHT/SCALE""",1760.0,"""1760""",null,97829080,57.21013,"""Female""","""Non-Hispanic/Latino""","""Black or African American""","""0""",686550527,2023-08-05 00:00:00,2023-10-03 00:00:00,2023-08-05 20:15:00,2023-08-05 23:21:00,2023-08-06 16:22:00,"""Yes""","""Inpatient""","""Emergency""","""Hospital Medicine""","""59""","""UH 07G""","""UH 10G""","""Emergency""","""Home & Outside Location""","""ED Admission""","""Bacteremia""","""Other cholangitis""","""Excision of Liver, Percutaneou…",0,"""NPOA-1""","""NPOA-1""",132.0,20.0,110.0,1.157407,335.674419,5.928571,68.9,8.737561,"""2023-08-05 20:50:00.0000000""","""Code Sepsis Order""","""0""","""0""","""0""","""0""","""0""","""0""",null,null,null,null,null,null,null
674101030,2023-01-08 20:45:00,"""Flowsheet""","""Pulse""","""PULSE""",81.0,"""81""",null,96080120,62.995208,"""Male""","""Non-Hispanic/Latino""","""White""","""0""",674101030,2022-12-16 00:00:00,2023-02-02 00:00:00,2022-12-16 18:01:00,2022-12-17 00:10:00,2022-12-16 18:01:00,"""No""","""Inpatient""","""Inpat

In [59]:
df_sepsis_3_with_shock = df_sepsis_3_with_shock.unique()

In [63]:
df_sepsis_3_with_shock.unique(subset=['EncounterEpicCsn', 'sepsis_instance'])

EncounterEpicCsn,infect_dt,Event_DateTime,sepsis_instance,sepsis_score,criterion,suspicion_infection_type,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,Baseline_Bilirubin,inr_criteria_flag,Baseline_PulseRate,Baseline_RespiratoryRate,Baseline_WBC,aptt_criteria_flag,coagulation_failure_flag,Baseline_SBP,platelets50_criteria_flag,Baseline_eGFR,Baseline_Platelets,Baseline_Creatinine,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,organ_dysfunction_total,sbp90_flag,sbpdelta40_flag,map65_flag,lactate4_flag,vasopressor_flag,septic_shock_flag,Event_DateTime_right,pf_ratio,vent_onoff_flag,vent_doc_flag,o2_vent_flag,vent_end_flag,pulmonary_dysfunction_flag
i64,datetime[μs],datetime[μs],datetime[μs],i32,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,f64,f64,f64,i32,i32,f64,i32,f64,f64,f64,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64,i64,datetime[μs],f64,i32,i32,i32,i32,i32
715402795,2024-11-07 17:21:00,2024-11-07 07:14:21,2024-11-07 07:14:21,3,"""culture_dt""","""IV+Culture""",96.6,68.0,24.0,51.0,31.7,22.11,13.96,4.0,0.6,1.3,280.0,1.2,null,14.0,0.03,0.2,0.3,null,0,0.428571,0,86.0,17.0,8.404915,0,0,127.0,0,6.930556,190.050847,10.861333,0,0,0,1,1,0,1,1,0,1,1,2024-11-07 04:19:00,114.0,null,null,null,null,1
696583672,2024-01-29 20:14:47,2024-01-29 00:35:00,2024-01-29 00:35:00,3,"""iv_dt""","""IV+Culture""",99.6,76.0,16.0,93.0,77.0,10.62,2.35,31.0,2.9,2.0,82.0,2.5,null,15.0,null,null,0.01,null,0,2.796154,1,78.0,16.0,6.060536,0,1,111.0,0,55.172131,125.75,1.51371,0,0,0,0,1,0,0,0,0,1,1,null,null,null,null,null,null,null
676469327,2023-02-07 10:27:50,2023-02-06 07:36:00,2023-02-06 07:36:00,3,"""iv_dt""","""IV+Culture""",97.7,79.0,17.0,125.0,78.3,13.95,7.15,6.0,null,0.8,115.0,null,null,11.0,null,null,0.11,null,0,1.2,0,null,null,7.16,0,0,147.0,0,8.0,176.0,null,0,1,0,1,2,0,0,0,0,1,1,2023-02-05 10:58:00,null,null,null,1,null,1
707544369,2024-07-18 02:32:00,2024-07-17 03:19:49.503,2024-07-17 03:19:49.503,3,"""iv_dt""","""IV+Culture""",99.1,85.0,24.0,92.0,82.0,32.72,2.38,25.0,null,1.9,175.0,null,null,13.0,null,null,0.01,null,0,0.1625,0,50.0,null,11.099167,0,0,112.0,0,59.5,184.166667,1.214286,0,1,0,1,2,0,0,0,0,1,1,null,null,null,null,null,null,null
686173502,2023-08-20 09:49:00,2023-08-18 10:45:00,2023-08-18 10:45:00,3,"""culture_dt""","""IV+Culture""",97.5,111.0,25.0,131.0,77.0,13.41,0.66,94.0,0.6,1.4,235.0,1.2,null,14.0,0.03,null,0.05,null,0,0.470588,0,100.0,18.0,6.079,0,0,105.0,0,105.033333,266.530612,0.546744,0,0,0,1,1,0,0,0,0,1,1,2023-08-27 21:21:00,100.0,null,null,null,null,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
671194516,2022-10-25 12:23:00,2022-10-25 08:15:00,2022-10-25 08:15:00,3,"""culture_dt""","""IV+Culture""",96.1,89.0,13.0,100.0,84.0,72.59,0.95,64.0,3.4,2.3,46.0,1.7,null,13.0,0.04,null,null,null,1,null,1,null,null,null,0,1,null,0,null,null,null,1,0,1,1,4,0,0,0,0,1,1,null,null,null,null,null,null,null
688651091,2024-01-19 16:39:00,2024-01-19 16:29:00,2024-01-19 16:29:00,3,"""culture_dt""","""IV+Culture""",96.2,103.0,26.0,110.0,93.3,37.6,1.2,68.0,7.4,2.0,192.0,1.3,null,11.0,0.04,null,0.01,0.04,0,0.85,0,72.0,16.0,7.33,0,0,139.0,0,81.666667,220.333333,1.03,0,0,1,1,2,0,0,0,0,1,1,null,null,null,null,null,null,null
665597637,2022-07-27 04:36:42,2022-07-25 19:15:00,2022-07-25 19:15:00,3,"""iv_dt""","""IV+Culture""",97.7,69.0,15.0,92.0,73.3,3.61,0.63,114.0,null,3.5,59.0,null,null,15.0,0.04,null,0.01,null,1,2.911538,0,96.0,16.0,7.183462,0,1,119.0,1,84.888889,301.82,1.015,0,0,0,0,2,0,0,0,0,1,1,null,null,null,null,null,null,null


In [ ]:
df_sepsis_3_with_shock.sort(by=['EncounterEpicCsn', 'sepsis_instance']).select(
    ['EncounterEpicCsn', 'infect_dt', 'Event_DateTime', pl.col("sepsis_instance").alias("sepsis_instant"), 'criterion', 'suspicion_infection_type']
    +
    [c for c in df_sepsis_3_with_shock.columns if c.startswith('last_') or c.endswith('_flag') or c.endswith('_score')]
).join

EncounterEpicCsn,infect_dt,Event_DateTime,sepsis_instant,criterion,suspicion_infection_type,sepsis_score,last_temp_8h,last_pulse_8h,last_resp_8h,last_sbp_8h,last_map_8h,last_wbc_12h,last_creatinine_12h,last_egfr_12h,last_bilirubin_12h,last_lactate_6h,last_platelets_24h,last_inr_12h,last_aptt_24h,last_gcs_12h,last_vasopressin_24h,last_phenylephrine_24h,last_norepinephrine_24h,last_epinephrine_24h,cardiovascular_failure_flag,inr_criteria_flag,aptt_criteria_flag,coagulation_failure_flag,platelets50_criteria_flag,platelets100_criteria_flag,renal_failure_flag,hepatic_failure_flag,neurological_failure_flag,sbp90_flag,sbpdelta40_flag,map65_flag,lactate4_flag,vasopressor_flag,septic_shock_flag
i64,datetime[μs],datetime[μs],datetime[μs],str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64
659308243,2022-06-14 22:10:00,2022-06-16 11:15:00,2022-06-14 22:10:00,"""lactate_dt""","""LACTATE+CULTURE""",3,97.9,71.0,26.0,118.0,74.0,11.62,3.35,19.0,0.8,1.8,86.0,null,null,9.0,0.04,null,0.03,null,0,0,0,0,0,0,1,0,1,0,0,0,0,1,1
659308243,2022-06-14 22:10:00,2022-06-16 09:45:00,2022-06-14 22:10:00,"""lactate_dt""","""LACTATE+CULTURE""",3,98.1,58.0,26.0,98.0,62.7,11.62,3.35,19.0,0.8,1.8,86.0,null,null,9.0,0.04,null,0.03,null,0,0,0,0,0,0,1,0,1,0,0,1,0,1,1
659308243,2022-06-14 22:10:00,2022-06-16 15:30:00,2022-06-14 22:10:00,"""lactate_dt""","""LACTATE+CULTURE""",3,98.9,82.0,29.0,127.0,78.3,11.59,3.27,20.0,0.8,2.1,82.0,null,null,9.0,0.04,null,0.03,null,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1
659308243,2022-06-14 22:10:00,2022-06-16 13:30:00,2022-06-14 22:10:00,"""lactate_dt""","""LACTATE+CULTURE""",3,97.9,60.0,26.0,116.0,71.3,11.59,3.27,20.0,0.8,2.1,82.0,null,null,9.0,0.04,null,0.03,null,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1
659308243,2022-06-14 22:10:00,2022-06-15 22:30:00,2022-06-14 22:10:00,"""lactate_dt""","""LACTATE+CULTURE""",3,98.4,84.0,27.0,118.0,78.7,17.79,2.72,24.0,null,2.8,145.0,1.3,null,3.0,0.04,null,0.22,null,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753558469,2026-05-25 19:43:52,2026-05-27 15:00:00,2026-05-25 19:43:52,"""iv_dt""","""IV+Culture""",3,99.0,106.0,26.0,89.0,70.3,18.04,1.64,38.0,null,3.0,252.0,null,null,8.0,null,null,0.01,null,1,0,0,0,0,0,1,0,1,0,0,0,0,1,1
753558469,2026-05-25 19:43:52,2026-05-27 13:15:00,2026-05-25 19:43:52,"""iv_dt""","""IV+Culture""",3,98.4,115.0,28.0,93.0,62.3,18.04,1.64,38.0,null,3.0,252.0,null,null,8.0,null,null,0.01,null,1,0,0,0,0,0,1,0,1,0,0,1,0,1,1
753558469,2026-05-25 19:43:52,2026-05-27 12:24:00,2026-05-25 19:43:52,"""iv_dt""","""IV+Culture""",3,98.4,119.0,29.0,93.0,78.3,18.04,1.64,38.0,null,3.0,252.0,1.6,null,8.0,null,null,0.01,null,1,1,0,1,0,0,1,0,1,0,0,0,0,1,1
